In [ ]:
'''
Author: Lucia Liu
Purpose: Gets summary tables from all images (eg. DDC 40x 1, DDC 40x 2, DDC 40x 3) and combines into one summary table w/ additional metrics
Last edited: 3/2/26
'''

In [8]:
import pandas as pd
from pathlib import Path
import os

In [ ]:
base_pathname = '/Users/lucialiu/Downloads/2025.12.16 Ctrl-SHKBP1 DDC diet 2wks p62/czi/40x/Ctrl-SHKBP1 Final Output/Control Final Output' # Replace with directory of folder w/ final image analysis outputs (from ImageJ)
combined_summary_pathname = '/Users/lucialiu/Downloads/2025.12.16 Ctrl-SHKBP1 DDC diet 2wks p62/czi/40x/Ctrl-SHKBP1 Final Output/Analysis Results/Ctrl_SHKBP1_summary.csv' # Replace with pathname of what you want to call your final combined summary table

In [9]:
results = pd.DataFrame(columns=[
    'Slice',
    'Diet',
    'Count C1',
    'Count C2',
    'Total Area C1',
    'Total Area C2',
    'Avg Particle Size',
    'Area of C1 / Area of C2',
    'Area of Particles (C1) / Cell'
])

In [10]:
def analyze_slice(pathname, slice, diet):
    df = pd.read_csv(pathname)
    df = df.set_index('Slice')

    c1 = df.loc[f'DUP_C1-{slice}.czi']
    c2 = df.loc[f'DUP_C2-{slice}.czi']

    data = {
        'Slice': slice,
        'Diet': diet,
        'Count C1': c1['Count'],
        'Count C2': c2['Count'],
        'Total Area C1': c1['Total Area'],
        'Total Area C2': c2['Total Area'],
        'Avg Particle Size': c1['Average Size'],
        'Area of C1 / Area of C2': (c1['Total Area'] / c2['Total Area']).round(4),
        'Area of Particles (C1) / Cell': (c1['Total Area'] / c2['Count']).round(4)
        #'Number of Particles (C1) / Cell': (c1['Count'] / c2['Count']).round(4)
    }
    results.loc[len(results)] = data

In [11]:
base_dir = Path(base_pathname)
summaries = {}
# Find summary file in each output folder
for path in base_dir.iterdir():
    if path.is_dir():
        for filename in os.listdir(base_pathname + '/' + path.name):
            if 'summary' in filename:
                summaries[filename.replace('_summary.csv', '')] = (base_pathname + '/' + path.name + '/' + filename)

summaries = dict(sorted(summaries.items()))

for slice, pathname in summaries.items():
    analyze_slice(pathname, slice, 'Chow')

#results = pd.concat([results.drop(index=8), results.loc[[8]]], ignore_index=True) # move DDC 40x 10 to end instead of after DDC 40x 1

results

,Slice,Diet,Count C1,Count C2,Total Area C1,Total Area C2,Avg Particle Size,Area of C1 / Area of C2,Area of Particles (C1) / Cell
0,Ctrl1_a,Chow,1374.0,89.0,1876.672,4137.720,1.366,0.4536,21.0862
1,Ctrl1_b,Chow,618.0,71.0,650.365,2210.551,1.052,0.2942,9.1601
2,Ctrl1_c,Chow,1063.0,70.0,934.705,2697.088,0.879,0.3466,13.3529
3,Ctrl1_d,Chow,1165.0,86.0,1804.111,3512.271,1.549,0.5137,20.9780
4,Ctrl2_a,Chow,561.0,91.0,1145.397,5750.162,2.042,0.1992,12.5868
5,Ctrl2_b,Chow,981.0,91.0,1463.734,3863.517,1.492,0.3789,16.0850
6,Ctrl2_c,Chow,512.0,76.0,391.185,3796.165,0.764,0.1030,5.1472
7,Ctrl3_a,Chow,690.0,93.0,585.505,8697.013,0.849,0.0673,6.2958
8,Ctrl3_b,Chow,460.0,81.0,521.236,3125.781,1.133,0.1668,6.4350
9,Ctrl3_c,Chow,1051.0,63.0,842.895,3141.366,0.802,0.2683,13.3793


In [14]:
results.to_csv(combined_summary_pathname, index=False)